In [2]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

device = torch.device("cpu")

print("Libraries imported successfully!")
print("Device:", device)

Libraries imported successfully!
Device: cpu


In [3]:
class SimpleDefenseEnv:
    def __init__(self, n_agents=2):
        self.n_agents = n_agents
        self.obs_dim = 1
        self.action_dim = 1

    def reset(self):
        self.positions = np.random.uniform(-5, 5, size=self.n_agents)
        return [np.array([p]) for p in self.positions]

    def step(self, actions):
        for i in range(self.n_agents):
            self.positions[i] += float(actions[i][0])

        dist_penalty = np.sum(np.abs(self.positions))
        reward = -dist_penalty

        rewards = [reward for _ in range(self.n_agents)]
        next_obs = [np.array([p]) for p in self.positions]

        done = False

        return next_obs, rewards, done

print("Environment class created successfully!")

Environment class created successfully!


In [4]:
env = SimpleDefenseEnv(n_agents=2)

obs = env.reset()

print("Number of agents:", env.n_agents)
print("Initial observations:", obs)

Number of agents: 2
Initial observations: [array([-1.50484358]), array([2.90280928])]


In [5]:
class Actor(nn.Module):
    def __init__(self, obs_dim, action_dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(obs_dim, 64),
            nn.ReLU(),

            nn.Linear(64, 64),
            nn.ReLU(),

            nn.Linear(64, action_dim),
            nn.Tanh()
        )

    def forward(self, obs):
        return self.net(obs)

print("Actor network created successfully!")

Actor network created successfully!


In [6]:
actor = Actor(obs_dim=1, action_dim=1)

test_obs = torch.tensor([[2.0]], dtype=torch.float32)

action = actor(test_obs)

print("Test observation:", test_obs.numpy())
print("Actor output:", action.detach().numpy())

Test observation: [[2.]]
Actor output: [[-0.44909927]]


In [7]:
class CentralizedCritic(nn.Module):
    def __init__(self, total_obs_dim, total_action_dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(total_obs_dim + total_action_dim, 128),
            nn.ReLU(),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 1)
        )

    def forward(self, joint_obs, joint_actions):
        x = torch.cat([joint_obs, joint_actions], dim=-1)
        return self.net(x)

print("Centralized Critic created successfully!")

Centralized Critic created successfully!


In [8]:
critic = CentralizedCritic(
    total_obs_dim=2,
    total_action_dim=2
)

joint_obs = torch.tensor([[1.0, -2.0]])
joint_actions = torch.tensor([[0.5, -0.5]])

q_value = critic(joint_obs, joint_actions)

print("Joint observations:", joint_obs.numpy())
print("Joint actions:", joint_actions.numpy())
print("Q-value:", q_value.detach().numpy())

Joint observations: [[ 1. -2.]]
Joint actions: [[ 0.5 -0.5]]
Q-value: [[-0.05805924]]


In [9]:
class MADDPGAgent:
    def __init__(
        self,
        obs_dim,
        action_dim,
        total_obs_dim,
        total_action_dim,
        lr=1e-3
    ):
        self.actor = Actor(obs_dim, action_dim).to(device)

        self.critic = CentralizedCritic(
            total_obs_dim,
            total_action_dim
        ).to(device)

        self.actor_optim = optim.Adam(
            self.actor.parameters(),
            lr=lr
        )

        self.critic_optim = optim.Adam(
            self.critic.parameters(),
            lr=lr
        )

    def select_action(self, obs, noise_scale=0.1):
        obs_t = torch.tensor(
            obs,
            dtype=torch.float32
        ).unsqueeze(0)

        action = self.actor(obs_t).detach().numpy()[0]

        action += noise_scale * np.random.randn(*action.shape)

        return np.clip(action, -1, 1)

print("MADDPG Agent class created successfully!")

MADDPG Agent class created successfully!


In [10]:
agent = MADDPGAgent(
    obs_dim=1,
    action_dim=1,
    total_obs_dim=2,
    total_action_dim=2
)

test_observation = np.array([2.0])

selected_action = agent.select_action(test_observation)

print("Observation:", test_observation)
print("Selected action:", selected_action)

Observation: [2.]
Selected action: [-0.13189508]


In [15]:
def train_maddpg(num_episodes=200, gamma=0.95):

    env = SimpleDefenseEnv(n_agents=2)

    n_agents = env.n_agents
    obs_dim = env.obs_dim
    action_dim = env.action_dim

    total_obs_dim = obs_dim * n_agents
    total_action_dim = action_dim * n_agents

    agents = [
        MADDPGAgent(
            obs_dim,
            action_dim,
            total_obs_dim,
            total_action_dim
        )
        for _ in range(n_agents)
    ]

    mse_loss = nn.MSELoss()

    for episode in range(num_episodes):

        obs = env.reset()

        actions = [
            agents[i].select_action(obs[i])
            for i in range(n_agents)
        ]

        next_obs, rewards, done = env.step(actions)

        joint_obs = torch.tensor(
            np.concatenate(obs),
            dtype=torch.float32
        ).unsqueeze(0)

        joint_next_obs = torch.tensor(
            np.concatenate(next_obs),
            dtype=torch.float32
        ).unsqueeze(0)

        joint_actions = torch.tensor(
            np.concatenate(actions),
            dtype=torch.float32
        ).unsqueeze(0)

        next_actions = []

        for i in range(n_agents):

            next_obs_i = torch.tensor(
                next_obs[i],
                dtype=torch.float32
            ).unsqueeze(0)

            next_actions.append(
                agents[i].actor(next_obs_i)
            )

        joint_next_actions = torch.cat(
            next_actions,
            dim=-1
        )

        for i in range(n_agents):

            with torch.no_grad():

                target_q = agents[i].critic(
                    joint_next_obs,
                    joint_next_actions
                )

                y = rewards[i] + gamma * target_q

            current_q = agents[i].critic(
                joint_obs,
                joint_actions
            )

            critic_loss = mse_loss(
                current_q,
                y
            )

            agents[i].critic_optim.zero_grad()

            critic_loss.backward()

            agents[i].critic_optim.step()

        for i in range(n_agents):

            obs_i = torch.tensor(
                obs[i],
                dtype=torch.float32
            ).unsqueeze(0)

            predicted_action_i = agents[i].actor(obs_i)

            actions_for_grad = []

            for j in range(n_agents):

                if j == i:

                    actions_for_grad.append(
                        predicted_action_i
                    )

                else:

                    a_j = torch.tensor(
                        actions[j],
                        dtype=torch.float32
                    ).unsqueeze(0)

                    actions_for_grad.append(a_j)

            joint_actions_grad = torch.cat(
                actions_for_grad,
                dim=-1
            )

            actor_loss = -agents[i].critic(
                joint_obs,
                joint_actions_grad
            ).mean()

            agents[i].actor_optim.zero_grad()

            actor_loss.backward()

            agents[i].actor_optim.step()

        if (episode + 1) % 20 == 0:

            avg_reward = np.mean(rewards)

            print(
                f"Episode {episode + 1}/{num_episodes} "
                f"| Avg Reward: {avg_reward:.3f}"
            )

    return agents

print("Training function created successfully!")

Training function created successfully!


In [16]:
trained_agents = train_maddpg(
    num_episodes=200
)

Episode 20/200 | Avg Reward: -1.846
Episode 40/200 | Avg Reward: -5.796
Episode 60/200 | Avg Reward: -7.586
Episode 80/200 | Avg Reward: -2.671
Episode 100/200 | Avg Reward: -4.249
Episode 120/200 | Avg Reward: -5.849
Episode 140/200 | Avg Reward: -3.043
Episode 160/200 | Avg Reward: -7.622
Episode 180/200 | Avg Reward: -7.005
Episode 200/200 | Avg Reward: -1.672


In [17]:
print("\nTraining complete.")

print("Number of trained agents:", len(trained_agents))

for i, agent in enumerate(trained_agents):
    print(f"\nAgent {i + 1}:")
    print(" - Decentralized Actor")
    print(" - Centralized Critic")


Training complete.
Number of trained agents: 2

Agent 1:
 - Decentralized Actor
 - Centralized Critic

Agent 2:
 - Decentralized Actor
 - Centralized Critic
